# Workbook 1 — Market trades: event time, dedup, OHLCV

Run in a Python kernel from the **repository root**, after `pip install -r requirements-dev.txt`. The fixture contains synthetic prices and repeats one trade.


In [ ]:
from pathlib import Path
import sys, json
from decimal import Decimal
root = Path.cwd()
if not (root / "tests/fixtures/trades.jsonl").exists():
    raise RuntimeError("Start Jupyter from the crypto_market_pulse repository root")
sys.path.insert(0, str(root / "src"))
from market_pulse.beam_pipeline import compute_stats
from market_pulse.domain import deduplicate, validate_event
from scripts.preview_marts import make_marts
fixture = [json.loads(s) for s in (root/"tests/fixtures/trades.jsonl").read_text().splitlines() if s.strip()]
len(fixture),len(deduplicate(fixture))


## A. Duplicates are physical events but not new trades
Check the duplicated `event_id` and compare arrival times.

In [ ]:
from collections import Counter
counts=Counter(e["event_id"] for e in fixture)
duplicates={k:v for k,v in counts.items() if v>1}
assert len(fixture)==8 and len(deduplicate(fixture))==7
duplicates


## B. Deterministic 1-minute market bars
Open/close uses `(event_time_ms, event_id)`, not file order.

In [ ]:
rows=make_marts(fixture)
for row in rows: print(row["symbol"],row["window_start"],row["trade_count"],row["open_price"],row["close_price"],row["vwap"])
assert len(rows)==4
btc=next(r for r in rows if r["symbol"]=="BTCUSDT" and r["window_start"].startswith("2026-10-30T10:00"))
assert btc["trade_count"]==4 and Decimal(btc["volume"])==Decimal("0.45")
btc


## C. Replay: invariance check
Repeat a trade and shuffle the input. `duplicate_count` changes, but OHLCV/VWAP remain the same.

In [ ]:
replayed = list(reversed(fixture)) + [fixture[0]]
orig = { (r["symbol"],r["window_start"]):r for r in make_marts(fixture)}
new = { (r["symbol"],r["window_start"]):r for r in make_marts(replayed)}
for key in orig:
    for metric in ("trade_count","open_price","close_price","volume","vwap"):
        assert orig[key][metric] == new[key][metric],(key,metric)
print("Replay invariant: PASS; extra physical duplicates:",sum(r["duplicate_count"] for r in new.values()))


## D. Questions
1. Why must VWAP use `Decimal` rather than floating-point prices?
2. Why is event-time sorting necessary for OHLC when packets arrive out of order?
3. When does query-level dedup become expensive on an infinite raw table?
4. What additional timestamp is needed for sink-to-dashboard latency?
